In [0]:
%sql
select * from datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated,customer_name_upper,Timestamp_Current,process_date
1001,2026-05-01,501,John Carter,john.carter@gmail.com,2001,Wireless Mouse,Electronics,25.99,2,51.98,USA,2026-05-07,JOHN CARTER,2026-05-08T06:42:31.016Z,2026-05-08
1002,2026-05-02,502,Emma Watson,emma.watson@yahoo.com,2002,Office Chair,Furniture,149.50,1,149.50,Canada,2026-05-07,EMMA WATSON,2026-05-08T06:42:31.016Z,2026-05-08
1003,2026-05-03,503,Rahul Sharma,rahul.sharma@gmail.com,2003,Mechanical Keyboard,Electronics,89.99,3,269.97,India,2026-05-07,RAHUL SHARMA,2026-05-08T06:42:31.016Z,2026-05-08
1004,2026-05-04,504,Sophia Lee,sophia.lee@outlook.com,2004,Running Shoes,Sports,79.99,2,159.98,Australia,2026-05-07,SOPHIA LEE,2026-05-08T06:42:31.016Z,2026-05-08
1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,129.99,1,129.99,Canada,2026-05-08,MICHAEL BROWN,2026-05-08T07:39:31.372Z,2026-05-08
1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,89.50,1,89.50,United Kingdom,2026-05-08,EMMA WILSON,2026-05-08T07:39:31.372Z,2026-05-08


In [0]:
spark.sql("create schema if not exists datamodeling.gold")

DataFrame[]

### Creating customer dimension table

In [0]:
%sql
select 
    distinct(customer_id),
    customer_name,
    customer_email,
    customer_name_upper
from datamodeling.silver.silver_table

customer_id,customer_name,customer_email,customer_name_upper
501,John Carter,john.carter@gmail.com,JOHN CARTER
503,Rahul Sharma,rahul.sharma@gmail.com,RAHUL SHARMA
504,Sophia Lee,sophia.lee@outlook.com,SOPHIA LEE
502,Emma Watson,emma.watson@yahoo.com,EMMA WATSON
506,Emma Wilson,emma.wilson@yahoo.com,EMMA WILSON
505,Michael Brown,michael.brown@gmail.com,MICHAEL BROWN


In [0]:
# Adding surrogate key which helps in performing joins
# First remove duplicates then only create the surrogate key

In [0]:
%sql
create or replace table datamodeling.gold.DimCustomers
using delta
as
with rem_dup as(
    select
        distinct(customer_id),
        customer_name,
        customer_email,
        customer_name_upper
    from datamodeling.silver.silver_table
)
select *,
    row_number() over(order by customer_id) as DimCustomerKey
from rem_dup

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.DimCustomers

customer_id,customer_name,customer_email,customer_name_upper,DimCustomerKey
501,John Carter,john.carter@gmail.com,JOHN CARTER,1
502,Emma Watson,emma.watson@yahoo.com,EMMA WATSON,2
503,Rahul Sharma,rahul.sharma@gmail.com,RAHUL SHARMA,3
504,Sophia Lee,sophia.lee@outlook.com,SOPHIA LEE,4
505,Michael Brown,michael.brown@gmail.com,MICHAEL BROWN,5
506,Emma Wilson,emma.wilson@yahoo.com,EMMA WILSON,6


### Creating Product Dimension table

In [0]:
%sql
create or replace table datamodeling.gold.DimProducts
using delta
as
with rem_dup as(
    select
        distinct(product_id),
        product_name,
        product_category
    from datamodeling.silver.silver_table
)
select *,
    row_number() over(order by product_id) as DimProductKey
from rem_dup

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.DimProducts

product_id,product_name,product_category,DimProductKey
2001,Wireless Mouse,Electronics,1
2002,Office Chair,Furniture,2
2003,Mechanical Keyboard,Electronics,3
2004,Running Shoes,Sports,4
2005,Wireless Headphones,Electronics,5
2006,Coffee Maker,Home Appliances,6


### Creating regions dimension table

In [0]:
%sql
create or replace table datamodeling.gold.DimRegion
using delta
as
with rem_dup as (
    select
        distinct(country)
    from datamodeling.silver.silver_table
)
select *,
    row_number() over(order by country) as DimRegionKey
from rem_dup

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.DimRegion

country,DimRegionKey
Australia,1
Canada,2
India,3
USA,4
United Kingdom,5


### Creating sales dimension table

- Here the silver_table provides the information of the sales
- So a separate sales dimension table containing all the dimensions has to be made

In [0]:
# Getting all the columns
spark.sql("select * from datamodeling.silver.silver_table").columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'customer_email',
 'product_id',
 'product_name',
 'product_category',
 'product_price',
 'quantity',
 'revenue',
 'country',
 'last_updated',
 'customer_name_upper',
 'Timestamp_Current',
 'process_date']

In [0]:
%sql
create or replace table datamodeling.gold.DimSales
using delta
as
select
    row_number() over(order by order_id) as DimSalesKey,
    order_id,
    order_date,
    customer_id,
    customer_name,
    customer_email,
    product_id,
    product_name,
    product_category,
    country,
    last_updated,
    customer_name_upper,
    Timestamp_Current,
    process_date
from datamodeling.silver.silver_table

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.DimSales

DimSalesKey,order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,country,last_updated,customer_name_upper,Timestamp_Current,process_date
1,1001,2026-05-01,501,John Carter,john.carter@gmail.com,2001,Wireless Mouse,Electronics,USA,2026-05-07,JOHN CARTER,2026-05-08T06:42:31.016Z,2026-05-08
2,1002,2026-05-02,502,Emma Watson,emma.watson@yahoo.com,2002,Office Chair,Furniture,Canada,2026-05-07,EMMA WATSON,2026-05-08T06:42:31.016Z,2026-05-08
3,1003,2026-05-03,503,Rahul Sharma,rahul.sharma@gmail.com,2003,Mechanical Keyboard,Electronics,India,2026-05-07,RAHUL SHARMA,2026-05-08T06:42:31.016Z,2026-05-08
4,1004,2026-05-04,504,Sophia Lee,sophia.lee@outlook.com,2004,Running Shoes,Sports,Australia,2026-05-07,SOPHIA LEE,2026-05-08T06:42:31.016Z,2026-05-08
5,1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,Canada,2026-05-08,MICHAEL BROWN,2026-05-08T07:39:31.372Z,2026-05-08
6,1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,United Kingdom,2026-05-08,EMMA WILSON,2026-05-08T07:39:31.372Z,2026-05-08


### Creating fact table
- Fact table is created by combining all the quantitative data from the original table and all the surrogate keys from the dimension tables
- The facts are obtained by joining the fact columns from the original table with dimension table columns(surrogate keys)

In [0]:
%sql
create or replace table datamodeling.gold.FactSales
using delta
as 
select
    S.DimSalesKey,
    P.DimProductKey,
    R.DimRegionKey,
    C.DimCustomerKey,
    F.product_price,
    F.quantity,
    F.revenue
from datamodeling.silver.silver_table as F
left join datamodeling.gold.DimSales as S
on F.order_id = S.order_id
left join datamodeling.gold.DimProducts as P
on F.product_id = P.product_id
left join datamodeling.gold.DimRegion as R
on F.country = R.country
left join datamodeling.gold.DimCustomers as C
on F.customer_id = C.customer_id

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from datamodeling.gold.factsales

DimSalesKey,DimProductKey,DimRegionKey,DimCustomerKey,product_price,quantity,revenue
1,1,4,1,25.99,2,51.98
2,2,2,2,149.50,1,149.50
3,3,3,3,89.99,3,269.97
4,4,1,4,79.99,2,159.98
5,5,2,5,129.99,1,129.99
6,6,5,6,89.50,1,89.50
